# 05. XGBoost

LightGBM과 같은 **그래디언트 부스팅 트리** 계열이라 강점과 약점도 같습니다.
여기서는 세 가지를 비교합니다.

1. **XGBoost 기본**
2. **단조 제약 + 얕은 트리**: 프로모션·할인·광고·공휴일은 판매를 *늘리기만*, 결품은 *줄이기만* 하도록 효과 방향을 강제
3. **하이브리드**: 선형 회귀로 추세·계절성을 먼저 빼고, 남은 부분만 XGBoost가 학습 (트리의 외삽 문제 해결)

In [1]:
import sys, warnings
sys.path.append("../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib  # 한글 글꼴
from common import *
plt.rcParams["figure.dpi"] = 110
df = load_data()
y = df.sellout.values.astype(float)
print(df.shape)

(104, 8)


In [2]:
t = np.arange(len(df), dtype=float)
ad = df.ad_spend.values.astype(float)
lag = lambda x, k: np.concatenate([np.zeros(k), x[:-k]])
F = pd.DataFrame(dict(
    t=t, s1=np.sin(2*np.pi*t/52), c1=np.cos(2*np.pi*t/52),
    disc=df.discount_pct.values, promo=df.promo.values,
    ad=ad, ad_l1=lag(ad, 1), ad_l2=lag(ad, 2), ad_l3=lag(ad, 3),   # 광고 이월은 지난 1~3주 값으로 스스로 배움
    hol=df.holiday.values, temp=df.temp.values, stock=df.stockout_days.values))
GROUP = {"t": "기본 수요", "s1": "계절성", "c1": "계절성", "disc": "할인", "promo": "프로모션",
         "ad": "광고", "ad_l1": "광고", "ad_l2": "광고", "ad_l3": "광고",
         "hol": "공휴일", "temp": "기온", "stock": "결품"}

def shap_to_contrib(shap):
    contrib = {f: np.zeros(len(df)) for f in FACTORS}
    contrib["기본 수요"] += shap[:, -1]          # 마지막 열 = 평균 예측(기준값)
    for j, c in enumerate(F.columns):
        contrib[GROUP[c]] += shap[:, j]
    return contrib
F.head()

,t,s1,c1,disc,promo,ad,ad_l1,ad_l2,ad_l3,hol,temp,stock
0,0.0,0.000000,1.000000,0,0,20.7,0.0,0.0,0.0,0,4.0,0
1,1.0,0.120537,0.992709,0,0,0.0,20.7,0.0,0.0,0,1.3,0
2,2.0,0.239316,0.970942,0,0,14.9,0.0,20.7,0.0,0,4.8,0
3,3.0,0.354605,0.935016,10,1,24.8,14.9,0.0,20.7,0,7.0,0
4,4.0,0.464723,0.885456,0,0,26.8,24.8,14.9,0.0,0,-0.1,0


In [3]:
FOLD_STARTS, FOLD_LEN = [72, 80, 88, 96], 8   # 앞부분으로 학습 → 다음 8주 예측, 4번 반복

def time_series_cv(fit_predict):
    """fit_predict(train_idx) → 전체 기간 예측. 모델이 보지 못한 다음 8주의 MAPE 평균"""
    errs = []
    for s0 in FOLD_STARTS:
        tr, te = np.arange(s0), np.arange(s0, min(s0 + FOLD_LEN, len(df)))
        errs.append(mape(y[te], fit_predict(tr)[te]))
    return np.mean(errs), np.round(errs, 1)

In [4]:
import xgboost as xgb

MONO = "(0,0,0,1,1,1,1,1,1,1,0,-1)"   # F 컬럼 순서: t,s1,c1,disc,promo,ad,ad_l1,ad_l2,ad_l3,hol,temp,stock
CONFIGS = {
    "XGBoost 기본": dict(max_depth=3),
    "단조 제약 + 얕은 트리": dict(max_depth=2, monotone_constraints=MONO, min_child_weight=3, reg_lambda=5),
}
def make(params):
    return xgb.XGBRegressor(n_estimators=400, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
                            random_state=7, **params)

def shap_to_contrib(shap):
    contrib = {f: np.zeros(len(df)) for f in FACTORS}
    contrib["기본 수요"] += shap[:, -1]
    for j, c in enumerate(F.columns):
        contrib[GROUP[c]] += shap[:, j]
    return contrib

rows, eff = [], {}
for name, params in CONFIGS.items():
    m = make(params).fit(F, y)
    p = m.predict(F)
    cv, folds = time_series_cv(lambda tr: make(params).fit(F.iloc[tr], y[tr]).predict(F))
    shap = m.get_booster().predict(xgb.DMatrix(F), pred_contribs=True)
    eff[name] = effects(shap_to_contrib(shap), df)
    rows.append(dict(모델=name, 학습MAPE=round(mape(y, p), 1), 새데이터MAPE=round(cv, 1)))
pd.DataFrame(rows)

,모델,학습MAPE,새데이터MAPE
0,XGBoost 기본,1.0,6.4
1,단조 제약 + 얕은 트리,2.5,6.1


## 하이브리드: 추세·계절은 선형으로, 나머지는 XGBoost로

트리가 못 하는 **추세 외삽**을 선형 회귀에 맡기고, 트리는 프로모션·할인 같은 요인 효과만 배우게 합니다.

In [5]:
TS = np.column_stack([np.ones(len(df)), t, np.sin(2*np.pi*t/52), np.cos(2*np.pi*t/52)])
F_rest = F.drop(columns=["t", "s1", "c1"])

def hybrid(tr):
    b = np.linalg.lstsq(TS[tr], y[tr], rcond=None)[0]           # 1단계: 추세 + 계절
    resid = y - TS @ b
    m = make(dict(max_depth=2, min_child_weight=3, reg_lambda=5)).fit(F_rest.iloc[tr], resid[tr])  # 2단계: 나머지
    return TS @ b + m.predict(F_rest)

p = hybrid(np.arange(len(df)))
cv, folds = time_series_cv(hybrid)
rows.append(dict(모델="하이브리드 (선형 추세 + XGBoost)", 학습MAPE=round(mape(y, p), 1), 새데이터MAPE=round(cv, 1)))
pd.DataFrame(rows)

,모델,학습MAPE,새데이터MAPE
0,XGBoost 기본,1.0,6.4
1,단조 제약 + 얕은 트리,2.5,6.1
2,하이브리드 (선형 추세 + XGBoost),2.6,5.6


In [6]:
effects_table(eff, df)

,정답,XGBoost 기본,단조 제약 + 얕은 트리
할인,14.0,11.3,11.4
프로모션,260.0,239.0,242.1
광고,52.9,41.0,45.9
공휴일,180.0,108.4,104.9
기온,-6.0,-0.7,-0.3
결품,-95.0,-74.3,-73.3
계절성,120.0,116.1,115.7
평균 오차율(%),NaN,28.9,28.8


## 결과 해석
- 기본 XGBoost는 LightGBM과 마찬가지로 **과적합**(학습 1%, 새 데이터 6%대)이고 공휴일·기온 효과를 작게 잡습니다.
- 단조 제약은 과적합 격차를 줄이지만 새 데이터 오차는 거의 그대로입니다. 주원인이 **추세 외삽 불가**이기 때문입니다.
- **하이브리드**는 추세를 선형 회귀가 맡아서 새 데이터 오차가 줄어들지만(6.4% → 5.6%), 선형 회귀(3.0%)에는 아직 못 미칩니다. 남은 오차는 트리가 104주 데이터로 프로모션·공휴일 효과를 충분히 배우지 못해서 생깁니다. 데이터가 많아지면 실무에서 자주 쓰는 방법입니다.

**결론**: 104주 데이터에서는 MMM·선형 회귀를 메인으로, XGBoost는 실제 데이터에 상호작용이 있는지 확인하는 보조 도구로 추천합니다.